## Example 1: shellingham functionality test 1

In [ ]:
# Example 1: Basic Shell Detection and Environment Analysis
import shellingham
import os
import platform
import shutil
import subprocess

def detect_real_shell():
    """shellingham.detect_shell() on a bash process this cell starts itself.

    Under the notebook runner the kernel has no shell among its ancestors (a Python process starts it),
    so detect_shell() on the kernel's own pid raises ShellDetectionFailure. The package's own `pid=`
    argument points detection at a real shell instead, which behaves the same wherever the kernel runs.
    The trailing `:` keeps bash alive as the process (it would otherwise exec `sleep` in its place), and
    detection waits for bash's "ready" line: right after spawn /proc/<pid>/cmdline can still be empty,
    and shellingham 1.5.4 raises TypeError on an empty command line.
    """
    bash = subprocess.Popen([shutil.which("bash"), "-c", "echo ready; sleep 60; :"], stdout=subprocess.PIPE, text=True)
    try:
        assert bash.stdout.readline().strip() == "ready", "bash did not start"
        return shellingham.detect_shell(pid=bash.pid)
    finally:
        bash.kill()
        bash.wait()
        bash.stdout.close()

print("=== Basic Shell Detection ===")

# Detect the shell of a real bash process (see detect_real_shell above)
shell_info = detect_real_shell()
shell_name, shell_path = shell_info
print(f"Detected shell: {shell_name}")
print(f"Shell path: {shell_path}")
print(f"Shell exists: {os.path.exists(shell_path) if shell_path else 'Unknown'}")
assert shell_name == 'bash' and os.path.isabs(shell_path) and os.path.exists(shell_path), shell_info

# System information
print(f"\n=== System Environment ===")
print(f"Operating system: {platform.system()}")
print(f"Platform: {platform.platform()}")
print(f"Python executable: {os.path.basename(os.sys.executable)}")

# Environment variables related to shell
shell_env_vars = [
    'SHELL', 'BASH', 'ZSH', 'FISH', 'TCSH', 'CSH',
    'COMSPEC', 'PSModulePath'  # Windows
]

print(f"\n=== Shell Environment Variables ===")
found_vars = {}
for var in shell_env_vars:
    value = os.environ.get(var)
    if value:
        found_vars[var] = value
        print(f"{var}: {value}")

if not found_vars:
    print("No shell environment variables found")

# Process information
print(f"\n=== Process Information ===")
print(f"Process ID: {os.getpid()}")
print(f"Parent Process ID: {os.getppid()}")

# Shell detection validation
print(f"\n=== Detection Validation ===")
if shell_name and shell_path:
    validation_checks = [
        ("Shell name is string", isinstance(shell_name, str)),
        ("Shell path is string", isinstance(shell_path, str)),
        ("Shell name not empty", bool(shell_name.strip())),
        ("Shell path not empty", bool(shell_path.strip())),
        ("Shell path is absolute", os.path.isabs(shell_path) if shell_path else False)
    ]
    
    for check_name, result in validation_checks:
        print(f"  {check_name}: {'✓' if result else '✗'}")
else:
    print("  No shell detected for validation")

# Shellingham module information
print(f"\n=== Module Information ===")
module_attrs = [attr for attr in dir(shellingham) if not attr.startswith('_')]
print(f"Public attributes: {module_attrs}")

# Common shell types
print(f"\n=== Common Shell Types ===")
common_shells = {
    'bash': 'Bourne Again Shell - most common Linux/macOS shell',
    'zsh': 'Z Shell - enhanced shell with better completion',
    'fish': 'Friendly Interactive Shell - user-friendly features',
    'tcsh': 'TENEX C Shell - enhanced C shell',
    'csh': 'C Shell - shell with C-like syntax',
    'cmd': 'Windows Command Prompt - Windows default',
    'powershell': 'PowerShell - Windows advanced shell',
    'pwsh': 'PowerShell Core - cross-platform PowerShell'
}

if shell_name:
    shell_lower = shell_name.lower()
    found_description = None
    
    for shell_type, description in common_shells.items():
        if shell_type in shell_lower:
            found_description = description
            break
    
    if found_description:
        print(f"Detected shell type: {found_description}")
    else:
        print(f"Shell '{shell_name}' - description not available")

print("Basic shell detection completed!")

## Example 2: shellingham functionality test 2

In [ ]:
# Example 2: Cross-Platform Shell Detection and Compatibility
import shellingham
import os
import subprocess
import platform
import shutil
import subprocess

def detect_real_shell():
    """shellingham.detect_shell() on a bash process this cell starts itself.

    Under the notebook runner the kernel has no shell among its ancestors (a Python process starts it),
    so detect_shell() on the kernel's own pid raises ShellDetectionFailure. The package's own `pid=`
    argument points detection at a real shell instead, which behaves the same wherever the kernel runs.
    The trailing `:` keeps bash alive as the process (it would otherwise exec `sleep` in its place), and
    detection waits for bash's "ready" line: right after spawn /proc/<pid>/cmdline can still be empty,
    and shellingham 1.5.4 raises TypeError on an empty command line.
    """
    bash = subprocess.Popen([shutil.which("bash"), "-c", "echo ready; sleep 60; :"], stdout=subprocess.PIPE, text=True)
    try:
        assert bash.stdout.readline().strip() == "ready", "bash did not start"
        return shellingham.detect_shell(pid=bash.pid)
    finally:
        bash.kill()
        bash.wait()
        bash.stdout.close()

print("=== Cross-Platform Shell Detection ===")

# Platform-specific detection strategies
print("=== Platform-Specific Analysis ===")

current_platform = platform.system()
print(f"Current platform: {current_platform}")

def analyze_platform_shells():
    """Analyze shell detection for current platform"""
    platform_info = {
        'Windows': {
            'default_shells': ['cmd', 'powershell', 'pwsh'],
            'env_vars': ['COMSPEC', 'PSModulePath'],
            'common_paths': ['C:\\Windows\\System32\\cmd.exe', 'C:\\Windows\\System32\\WindowsPowerShell\\v1.0\\powershell.exe']
        },
        'Linux': {
            'default_shells': ['bash', 'zsh', 'fish', 'dash'],
            'env_vars': ['SHELL', 'BASH', 'ZSH'],
            'common_paths': ['/bin/bash', '/usr/bin/bash', '/bin/zsh', '/usr/bin/zsh']
        },
        'Darwin': {  # macOS
            'default_shells': ['bash', 'zsh', 'fish'],
            'env_vars': ['SHELL', 'BASH', 'ZSH'],
            'common_paths': ['/bin/bash', '/bin/zsh', '/usr/local/bin/fish']
        }
    }
    
    current_info = platform_info.get(current_platform, {
        'default_shells': ['unknown'],
        'env_vars': ['SHELL'],
        'common_paths': []
    })
    
    print(f"Expected shells for {current_platform}:")
    for shell in current_info['default_shells']:
        print(f"  - {shell}")
    
    print(f"Relevant environment variables:")
    for var in current_info['env_vars']:
        value = os.environ.get(var, 'Not set')
        print(f"  {var}: {value}")
    
    print(f"Common shell paths to check:")
    for path in current_info['common_paths']:
        exists = os.path.exists(path)
        print(f"  {path}: {'✓' if exists else '✗'}")
    
    return current_info

platform_shells = analyze_platform_shells()

# Test detection across different methods
print(f"\n=== Detection Method Comparison ===")

detection_methods = []

# Method 1: Shellingham detection (on a real bash process, see detect_real_shell above)
shell_result = detect_real_shell()
assert shell_result[0] == 'bash', shell_result
detection_methods.append({
    'method': 'shellingham.detect_shell(pid=<bash>)',
    'result': shell_result,
    'success': shell_result is not None,
    'details': f"Shell: {shell_result[0]}, Path: {shell_result[1]}" if shell_result else "No detection"
})

# Method 2: Environment variable detection
shell_env = os.environ.get('SHELL')
if shell_env:
    shell_name = os.path.basename(shell_env)
    detection_methods.append({
        'method': 'Environment SHELL variable',
        'result': (shell_name, shell_env),
        'success': True,
        'details': f"Shell: {shell_name}, Path: {shell_env}"
    })
else:
    detection_methods.append({
        'method': 'Environment SHELL variable',
        'result': None,
        'success': False,
        'details': "SHELL variable not set"
    })

# Method 3: Process parent detection (simplified)
try:
    parent_pid = os.getppid()
    detection_methods.append({
        'method': 'Parent process analysis',
        'result': f"PPID: {parent_pid}",
        'success': True,
        'details': f"Parent process ID: {parent_pid}"
    })
except Exception as e:
    detection_methods.append({
        'method': 'Parent process analysis',
        'result': None,
        'success': False,
        'details': f"Error: {e}"
    })
    raise

# Show detection results
for method in detection_methods:
    print(f"{method['method']}:")
    print(f"  Success: {'✓' if method['success'] else '✗'}")
    print(f"  Details: {method['details']}")

# Shell compatibility testing
print(f"\n=== Shell Compatibility Testing ===")

def test_shell_compatibility():
    """Test compatibility with detected shell"""
    detected = detect_real_shell()
    assert detected[0] == 'bash', detected
    
    if detected:
        shell_name, shell_path = detected
        print(f"Testing compatibility for: {shell_name}")
        
        # Basic compatibility tests
        compatibility_tests = [
            ("Shell executable exists", os.path.exists(shell_path) if shell_path else False),
            ("Shell executable is file", os.path.isfile(shell_path) if shell_path else False),
            ("Shell path is absolute", os.path.isabs(shell_path) if shell_path else False),
            ("Shell name is recognized", shell_name.lower() in [s.lower() for s in platform_shells.get('default_shells', [])])
        ]
        
        for test_name, result in compatibility_tests:
            print(f"  {test_name}: {'✓' if result else '✗'}")
        
        # Shell-specific features
        shell_features = {
            'bash': ['command completion', 'history expansion', 'job control'],
            'zsh': ['advanced completion', 'themes', 'plugins'],
            'fish': ['syntax highlighting', 'web-based config', 'smart suggestions'],
            'cmd': ['batch scripting', 'environment variables', 'command history'],
            'powershell': ['object pipeline', 'cmdlets', '.NET integration']
        }
        
        shell_lower = shell_name.lower()
        features = None
        for shell_type in shell_features:
            if shell_type in shell_lower:
                features = shell_features[shell_type]
                break
        
        if features:
            print(f"  Key features of {shell_name}:")
            for feature in features:
                print(f"    - {feature}")
    else:
        print("No shell detected for compatibility testing")

test_shell_compatibility()

print("Cross-platform shell detection completed!")

## Example 3: shellingham functionality test 3

In [ ]:
# Example 3: Shell Detection Algorithms and Process Analysis
import shellingham
import os
import platform
import shutil
import subprocess

def detect_real_shell():
    """shellingham.detect_shell() on a bash process this cell starts itself.

    Under the notebook runner the kernel has no shell among its ancestors (a Python process starts it),
    so detect_shell() on the kernel's own pid raises ShellDetectionFailure. The package's own `pid=`
    argument points detection at a real shell instead, which behaves the same wherever the kernel runs.
    The trailing `:` keeps bash alive as the process (it would otherwise exec `sleep` in its place), and
    detection waits for bash's "ready" line: right after spawn /proc/<pid>/cmdline can still be empty,
    and shellingham 1.5.4 raises TypeError on an empty command line.
    """
    bash = subprocess.Popen([shutil.which("bash"), "-c", "echo ready; sleep 60; :"], stdout=subprocess.PIPE, text=True)
    try:
        assert bash.stdout.readline().strip() == "ready", "bash did not start"
        return shellingham.detect_shell(pid=bash.pid)
    finally:
        bash.kill()
        bash.wait()
        bash.stdout.close()

print("=== Shell Detection Algorithms ===")

# Simulate different detection strategies
print("=== Detection Strategy Analysis ===")

class ShellDetectionAnalyzer:
    """Analyze different shell detection strategies"""
    
    def __init__(self):
        self.strategies = [
            'environment_variables',
            'process_tree_analysis', 
            'platform_defaults',
            'path_scanning'
        ]
        self.results = {}
    
    def analyze_environment_variables(self):
        """Analyze environment variable-based detection"""
        env_results = {}
        
        # Primary shell variables
        primary_vars = {
            'SHELL': os.environ.get('SHELL'),
            'BASH': os.environ.get('BASH'),
            'ZSH': os.environ.get('ZSH')
        }
        
        # Windows-specific variables
        windows_vars = {
            'COMSPEC': os.environ.get('COMSPEC'),
            'PSModulePath': os.environ.get('PSModulePath')
        }
        
        # Terminal variables
        terminal_vars = {
            'TERM': os.environ.get('TERM'),
            'TERM_PROGRAM': os.environ.get('TERM_PROGRAM'),
            'TERMINAL_EMULATOR': os.environ.get('TERMINAL_EMULATOR')
        }
        
        all_vars = {**primary_vars, **windows_vars, **terminal_vars}
        
        found_vars = {k: v for k, v in all_vars.items() if v}
        env_results['found_variables'] = found_vars
        env_results['variable_count'] = len(found_vars)
        
        # Determine likely shell from environment
        likely_shell = None
        if primary_vars['SHELL']:
            likely_shell = os.path.basename(primary_vars['SHELL'])
        elif windows_vars['COMSPEC']:
            likely_shell = os.path.basename(windows_vars['COMSPEC'])
        elif windows_vars['PSModulePath']:
            likely_shell = 'powershell'
        
        env_results['likely_shell'] = likely_shell
        self.results['environment_variables'] = env_results
        
        return env_results
    
    def analyze_process_information(self):
        """Analyze process-based detection"""
        process_results = {}
        
        try:
            process_results['current_pid'] = os.getpid()
            process_results['parent_pid'] = os.getppid()
            process_results['process_detection_available'] = True
            
            # Working directory analysis
            process_results['working_directory'] = os.getcwd()
            process_results['python_executable'] = os.sys.executable
            
        except Exception as e:
            process_results['process_detection_available'] = False
            process_results['error'] = str(e)
        
        self.results['process_analysis'] = process_results
        return process_results
    
    def analyze_platform_defaults(self):
        """Analyze platform-specific default shells"""
        platform_results = {}
        
        current_platform = platform.system()
        platform_results['system'] = current_platform
        platform_results['platform_full'] = platform.platform()
        
        # Platform-specific default shells
        defaults = {
            'Windows': ['cmd.exe', 'powershell.exe', 'pwsh.exe'],
            'Linux': ['bash', 'dash', 'sh'],
            'Darwin': ['bash', 'zsh'],  # macOS changed default to zsh in Catalina
            'FreeBSD': ['sh', 'tcsh'],
            'OpenBSD': ['ksh', 'sh']
        }
        
        platform_results['expected_defaults'] = defaults.get(current_platform, ['sh'])
        
        # Check for version-specific changes (e.g., macOS Catalina+ uses zsh)
        if current_platform == 'Darwin':
            try:
                mac_version = platform.mac_ver()[0]
                if mac_version:
                    major_version = int(mac_version.split('.')[0])
                    if major_version >= 10:  # macOS 10.15+ (Catalina+)
                        platform_results['default_shell'] = 'zsh'
                    else:
                        platform_results['default_shell'] = 'bash'
                else:
                    platform_results['default_shell'] = 'unknown'
            except:
                platform_results['default_shell'] = 'bash'  # fallback
        
        self.results['platform_defaults'] = platform_results
        return platform_results
    
    def analyze_path_scanning(self):
        """Analyze path-based shell detection"""
        path_results = {}
        
        # Common shell locations by platform
        shell_paths = {
            'Unix': [
                '/bin/bash', '/usr/bin/bash', '/usr/local/bin/bash',
                '/bin/zsh', '/usr/bin/zsh', '/usr/local/bin/zsh',
                '/bin/fish', '/usr/bin/fish', '/usr/local/bin/fish',
                '/bin/sh', '/usr/bin/sh',
                '/bin/tcsh', '/usr/bin/tcsh',
                '/bin/csh', '/usr/bin/csh'
            ],
            'Windows': [
                'C:\\Windows\\System32\\cmd.exe',
                'C:\\Windows\\System32\\WindowsPowerShell\\v1.0\\powershell.exe',
                'C:\\Program Files\\PowerShell\\7\\pwsh.exe'
            ]
        }
        
        current_platform = platform.system()
        if current_platform in ['Linux', 'Darwin', 'FreeBSD', 'OpenBSD']:
            paths_to_check = shell_paths['Unix']
        elif current_platform == 'Windows':
            paths_to_check = shell_paths['Windows']
        else:
            paths_to_check = shell_paths['Unix']  # fallback
        
        found_shells = []
        for path in paths_to_check:
            if os.path.exists(path):
                shell_name = os.path.basename(path)
                found_shells.append({
                    'name': shell_name,
                    'path': path,
                    'is_executable': os.access(path, os.X_OK) if os.path.exists(path) else False
                })
        
        path_results['searched_paths'] = len(paths_to_check)
        path_results['found_shells'] = found_shells
        path_results['found_count'] = len(found_shells)
        
        self.results['path_scanning'] = path_results
        return path_results
    
    def run_full_analysis(self):
        """Run all detection analyses"""
        print("Running comprehensive shell detection analysis...")
        
        env_results = self.analyze_environment_variables()
        process_results = self.analyze_process_information()
        platform_results = self.analyze_platform_defaults()
        path_results = self.analyze_path_scanning()
        
        return self.results

# Run detection analysis
analyzer = ShellDetectionAnalyzer()
analysis_results = analyzer.run_full_analysis()

# Display results
print("\n=== Environment Variables Analysis ===")
env_data = analysis_results['environment_variables']
print(f"Variables found: {env_data['variable_count']}")
print(f"Likely shell: {env_data['likely_shell'] or 'Unknown'}")
for var, value in env_data['found_variables'].items():
    print(f"  {var}: {value}")

print("\n=== Process Analysis ===")
process_data = analysis_results['process_analysis']
# os.getpid/getppid/getcwd cannot legitimately fail here: a failure is a broken example, not a result to print
assert process_data['process_detection_available'], process_data.get('error', 'Unknown error')
print(f"Current PID: {process_data['current_pid']}")
print(f"Parent PID: {process_data['parent_pid']}")
print(f"Working directory: {process_data['working_directory']}")

print("\n=== Platform Defaults Analysis ===")
platform_data = analysis_results['platform_defaults']
print(f"Platform: {platform_data['system']}")
print(f"Expected default shells: {platform_data['expected_defaults']}")
if 'default_shell' in platform_data:
    print(f"Platform default shell: {platform_data['default_shell']}")

print("\n=== Path Scanning Analysis ===")
path_data = analysis_results['path_scanning']
print(f"Paths searched: {path_data['searched_paths']}")
print(f"Shells found: {path_data['found_count']}")
for shell in path_data['found_shells']:
    executable_status = "executable" if shell['is_executable'] else "not executable"
    print(f"  {shell['name']}: {shell['path']} ({executable_status})")

# Compare with shellingham results
print("\n=== Shellingham vs Analysis Comparison ===")
shellingham_result = detect_real_shell()  # a real bash process, see detect_real_shell above
detected_name, detected_path = shellingham_result
print(f"Shellingham detected: {detected_name} at {detected_path}")
assert detected_name == 'bash' and os.path.exists(detected_path), shellingham_result

# Cross-validate with analysis
env_likely = analysis_results['environment_variables']['likely_shell']
path_found = any(shell['name'] == detected_name for shell in path_data['found_shells'])

print(f"Cross-validation:")
print(f"  Matches environment analysis: {detected_name == env_likely}")
print(f"  Found in path scanning: {path_found}")
print(f"  Path exists: {os.path.exists(detected_path) if detected_path else False}")

print("Shell detection algorithms analysis completed!")

## Example 4: shellingham functionality test 4

In [ ]:
# Example 4: Shell Integration and Application Development
import shellingham
import os
import shlex
import shutil
import subprocess

def detect_real_shell():
    """shellingham.detect_shell() on a bash process this cell starts itself.

    Under the notebook runner the kernel has no shell among its ancestors (a Python process starts it),
    so detect_shell() on the kernel's own pid raises ShellDetectionFailure. The package's own `pid=`
    argument points detection at a real shell instead, which behaves the same wherever the kernel runs.
    The trailing `:` keeps bash alive as the process (it would otherwise exec `sleep` in its place), and
    detection waits for bash's "ready" line: right after spawn /proc/<pid>/cmdline can still be empty,
    and shellingham 1.5.4 raises TypeError on an empty command line.
    """
    bash = subprocess.Popen([shutil.which("bash"), "-c", "echo ready; sleep 60; :"], stdout=subprocess.PIPE, text=True)
    try:
        assert bash.stdout.readline().strip() == "ready", "bash did not start"
        return shellingham.detect_shell(pid=bash.pid)
    finally:
        bash.kill()
        bash.wait()
        bash.stdout.close()

print("=== Shell Integration and Application Development ===")

# Application shell detection utility
class ApplicationShellManager:
    """Manage shell detection and integration for applications"""
    
    def __init__(self, app_name="demo_app"):
        self.app_name = app_name
        self.detected_shell = None
        self.shell_features = {}
        self.initialize_shell_detection()
    
    def initialize_shell_detection(self):
        """Initialize shell detection for the application"""
        try:
            shell_info = detect_real_shell()  # a real bash process, see detect_real_shell above
            if shell_info:
                self.detected_shell = {
                    'name': shell_info[0],
                    'path': shell_info[1],
                    'available': True
                }
                self.analyze_shell_features()
            else:
                self.detected_shell = {'available': False}
        except shellingham.ShellDetectionFailure as e:
            # the package's documented "no shell found" signal; anything else is a real failure
            self.detected_shell = {'available': False, 'error': repr(e)}
    
    def analyze_shell_features(self):
        """Analyze features of the detected shell"""
        if not self.detected_shell or not self.detected_shell.get('available'):
            return
        
        shell_name = self.detected_shell['name'].lower()
        
        # Shell-specific feature analysis
        feature_sets = {
            'bash': {
                'completion': True,
                'history': True,
                'aliases': True,
                'functions': True,
                'job_control': True,
                'scripting': 'bash',
                'config_files': ['.bashrc', '.bash_profile', '.bash_aliases']
            },
            'zsh': {
                'completion': True,
                'history': True,
                'aliases': True,
                'functions': True,
                'job_control': True,
                'scripting': 'zsh',
                'themes': True,
                'plugins': True,
                'config_files': ['.zshrc', '.zsh_profile', '.oh-my-zsh']
            },
            'fish': {
                'completion': True,
                'history': True,
                'aliases': False,  # fish uses functions instead
                'functions': True,
                'job_control': True,
                'scripting': 'fish',
                'syntax_highlighting': True,
                'web_config': True,
                'config_files': ['config.fish', 'functions/']
            },
            'cmd': {
                'completion': False,
                'history': True,
                'aliases': False,
                'functions': False,
                'job_control': False,
                'scripting': 'batch',
                'config_files': []
            },
            'powershell': {
                'completion': True,
                'history': True,
                'aliases': True,
                'functions': True,
                'job_control': True,
                'scripting': 'powershell',
                'objects': True,
                'cmdlets': True,
                'config_files': ['$PROFILE']
            }
        }
        
        # Find matching features
        for shell_type, features in feature_sets.items():
            if shell_type in shell_name:
                self.shell_features = features
                break
        
        if not self.shell_features:
            self.shell_features = {'completion': False, 'scripting': 'unknown'}
    
    def get_shell_info(self):
        """Get comprehensive shell information"""
        info = {
            'application': self.app_name,
            'detection_successful': self.detected_shell.get('available', False),
        }
        
        if self.detected_shell.get('available'):
            info.update({
                'shell_name': self.detected_shell['name'],
                'shell_path': self.detected_shell['path'],
                'features': self.shell_features
            })
        else:
            info['error'] = self.detected_shell.get('error', 'Detection failed')
        
        return info
    
    def generate_shell_command(self, command, shell_safe=True):
        """Generate shell-appropriate command"""
        if not self.detected_shell.get('available'):
            return None
        
        shell_name = self.detected_shell['name'].lower()
        
        if shell_safe:
            # Use shlex for safe shell escaping
            if 'cmd' in shell_name:
                # Windows cmd has different escaping rules
                return f'"{command}"'
            else:
                # Unix-like shells
                return shlex.quote(command)
        
        return command
    
    def get_config_file_paths(self):
        """Get configuration file paths for the detected shell"""
        if not self.shell_features.get('config_files'):
            return []
        
        home_dir = os.path.expanduser('~')
        config_paths = []
        
        for config_file in self.shell_features['config_files']:
            if config_file.startswith('$'):
                # Handle special variables like $PROFILE
                continue
            
            full_path = os.path.join(home_dir, config_file)
            config_paths.append({
                'file': config_file,
                'path': full_path,
                'exists': os.path.exists(full_path)
            })
        
        return config_paths

# Test application shell management
print("=== Application Shell Management ===")

app_shell = ApplicationShellManager("shell_detection_demo")
shell_info = app_shell.get_shell_info()
assert shell_info['detection_successful'] and shell_info['shell_name'] == 'bash', shell_info

print(f"Application: {shell_info['application']}")
print(f"Detection successful: {shell_info['detection_successful']}")

if shell_info['detection_successful']:
    print(f"Shell: {shell_info['shell_name']}")
    print(f"Path: {shell_info['shell_path']}")
    
    # Show features
    print(f"\nShell features:")
    for feature, supported in shell_info['features'].items():
        if isinstance(supported, bool):
            print(f"  {feature}: {'✓' if supported else '✗'}")
        else:
            print(f"  {feature}: {supported}")
    
    # Configuration files
    config_files = app_shell.get_config_file_paths()
    if config_files:
        print(f"\nConfiguration files:")
        for config in config_files:
            status = "exists" if config['exists'] else "not found"
            print(f"  {config['file']}: {status}")

# Command generation examples
print(f"\n=== Command Generation Examples ===")

test_commands = [
    "echo Hello World",
    "python script.py --arg value",
    "ls -la | grep .txt",
    "cd /path/with spaces/"
]

for cmd in test_commands:
    safe_cmd = app_shell.generate_shell_command(cmd, shell_safe=True)
    if safe_cmd:
        print(f"Original: {cmd}")
        print(f"Safe:     {safe_cmd}")
        print()

# Integration patterns for different use cases
print(f"=== Integration Patterns ===")

integration_patterns = {
    'CLI_Application': {
        'description': 'Command-line tool that needs shell integration',
        'use_case': 'Auto-completion, shell aliases, command history',
        'implementation': 'Detect shell and generate appropriate completion scripts'
    },
    'IDE_Terminal': {
        'description': 'IDE with integrated terminal',
        'use_case': 'Launch terminal with correct shell, shell-aware features',
        'implementation': 'Use detected shell for terminal initialization'
    },
    'Build_Tool': {
        'description': 'Build system executing shell commands',
        'use_case': 'Execute commands in user\'s preferred shell',
        'implementation': 'Shell detection for cross-platform compatibility'
    },
    'Package_Manager': {
        'description': 'Package manager with shell integration',
        'use_case': 'Shell completion, environment activation',
        'implementation': 'Generate shell-specific activation scripts'
    }
}

for pattern_name, details in integration_patterns.items():
    print(f"{pattern_name.replace('_', ' ')}:")
    print(f"  Description: {details['description']}")
    print(f"  Use case: {details['use_case']}")
    print(f"  Implementation: {details['implementation']}")
    print()

# Best practices for shell detection in applications
print(f"=== Best Practices ===")

best_practices = [
    "Always provide fallback behavior when shell detection fails",
    "Cache shell detection results to avoid repeated system calls",
    "Respect user's shell preferences and configuration",
    "Use appropriate escaping for shell command generation",
    "Test shell integration across different platforms",
    "Provide options to override automatic shell detection",
    "Handle shell-specific features gracefully when unavailable",
    "Document shell requirements and supported shells"
]

for i, practice in enumerate(best_practices, 1):
    print(f"{i}. {practice}")

print("Shell integration and application development completed!")

## Example 5: shellingham functionality test 5

In [ ]:
# Example 5: Production Usage and Development Tools Integration
import shellingham
import os
import json
from pathlib import Path
import shutil
import subprocess

def detect_real_shell():
    """shellingham.detect_shell() on a bash process this cell starts itself.

    Under the notebook runner the kernel has no shell among its ancestors (a Python process starts it),
    so detect_shell() on the kernel's own pid raises ShellDetectionFailure. The package's own `pid=`
    argument points detection at a real shell instead, which behaves the same wherever the kernel runs.
    The trailing `:` keeps bash alive as the process (it would otherwise exec `sleep` in its place), and
    detection waits for bash's "ready" line: right after spawn /proc/<pid>/cmdline can still be empty,
    and shellingham 1.5.4 raises TypeError on an empty command line.
    """
    bash = subprocess.Popen([shutil.which("bash"), "-c", "echo ready; sleep 60; :"], stdout=subprocess.PIPE, text=True)
    try:
        assert bash.stdout.readline().strip() == "ready", "bash did not start"
        return shellingham.detect_shell(pid=bash.pid)
    finally:
        bash.kill()
        bash.wait()
        bash.stdout.close()

print("=== Production Usage and Development Tools Integration ===")

# Production-ready shell detection service
class ProductionShellDetector:
    """Production-ready shell detection with caching and error handling"""
    
    def __init__(self, cache_duration=300):  # 5 minutes cache
        self.cache_duration = cache_duration
        self.cache = {}
        self.supported_shells = {
            'bash': {'priority': 1, 'features': ['completion', 'history', 'scripting']},
            'zsh': {'priority': 1, 'features': ['completion', 'history', 'scripting', 'themes']},
            'fish': {'priority': 2, 'features': ['completion', 'history', 'syntax_highlighting']},
            'cmd': {'priority': 3, 'features': ['basic']},
            'powershell': {'priority': 2, 'features': ['completion', 'objects', 'scripting']},
            'pwsh': {'priority': 2, 'features': ['completion', 'objects', 'scripting']},
            'tcsh': {'priority': 4, 'features': ['completion', 'history']},
            'csh': {'priority': 4, 'features': ['basic']},
            'sh': {'priority': 5, 'features': ['basic']}
        }
    
    def detect_with_fallback(self):
        """Detect shell with multiple fallback strategies"""
        detection_strategies = [
            self._shellingham_detection,
            self._environment_detection,
            self._platform_default_detection
        ]
        
        for strategy in detection_strategies:
            try:
                result = strategy()
                if result:
                    return {
                        'shell_name': result[0],
                        'shell_path': result[1],
                        'detection_method': strategy.__name__,
                        'success': True
                    }
            except shellingham.ShellDetectionFailure:
                continue  # the package's documented "no shell found" signal: fall through to the next strategy
        
        return {'success': False, 'error': 'All detection strategies failed'}
    
    def _shellingham_detection(self):
        """Primary detection using shellingham (on a real bash process, see detect_real_shell above)"""
        return detect_real_shell()
    
    def _environment_detection(self):
        """Fallback detection using environment variables"""
        shell_env = os.environ.get('SHELL')
        if shell_env:
            return (os.path.basename(shell_env), shell_env)
        
        # Windows fallback
        comspec = os.environ.get('COMSPEC')
        if comspec:
            return (os.path.basename(comspec), comspec)
        
        return None
    
    def _platform_default_detection(self):
        """Final fallback using platform defaults"""
        import platform
        system = platform.system()
        
        defaults = {
            'Windows': ('cmd.exe', 'C:\\Windows\\System32\\cmd.exe'),
            'Linux': ('bash', '/bin/bash'),
            'Darwin': ('zsh', '/bin/zsh')  # macOS default since Catalina
        }
        
        return defaults.get(system)
    
    def get_shell_priority(self, shell_name):
        """Get priority rating for shell (1=highest, 5=lowest)"""
        shell_lower = shell_name.lower()
        for shell, info in self.supported_shells.items():
            if shell in shell_lower:
                return info['priority']
        return 5  # Unknown shell, lowest priority
    
    def get_shell_capabilities(self, shell_name):
        """Get capabilities of detected shell"""
        shell_lower = shell_name.lower()
        for shell, info in self.supported_shells.items():
            if shell in shell_lower:
                return {
                    'shell': shell,
                    'priority': info['priority'],
                    'features': info['features'],
                    'recommended': info['priority'] <= 2
                }
        
        return {
            'shell': 'unknown',
            'priority': 5,
            'features': ['basic'],
            'recommended': False
        }

# Test production shell detection
print("=== Production Shell Detection ===")

prod_detector = ProductionShellDetector()
detection_result = prod_detector.detect_with_fallback()
assert detection_result['success'] and detection_result['detection_method'] == '_shellingham_detection', detection_result
assert detection_result['shell_name'] == 'bash', detection_result

if detection_result['success']:
    shell_name = detection_result['shell_name']
    shell_path = detection_result['shell_path']
    method = detection_result['detection_method']
    
    print(f"Shell detected: {shell_name}")
    print(f"Path: {shell_path}")
    print(f"Method: {method}")
    
    # Get capabilities
    capabilities = prod_detector.get_shell_capabilities(shell_name)
    print(f"\nShell capabilities:")
    print(f"  Type: {capabilities['shell']}")
    print(f"  Priority: {capabilities['priority']} (1=best, 5=worst)")
    print(f"  Features: {', '.join(capabilities['features'])}")
    print(f"  Recommended: {capabilities['recommended']}")

# Development tools integration patterns
print(f"\n=== Development Tools Integration ===")

class DevToolsShellIntegration:
    """Integration patterns for common development tools"""
    
    def __init__(self, shell_detector):
        self.detector = shell_detector
        self.integrations = {}
    
    def generate_cli_completion(self, app_name, commands):
        """Generate shell completion for CLI application"""
        detection = self.detector.detect_with_fallback()
        if not detection['success']:
            return None
        
        shell_name = detection['shell_name'].lower()
        
        if 'bash' in shell_name:
            return self._generate_bash_completion(app_name, commands)
        elif 'zsh' in shell_name:
            return self._generate_zsh_completion(app_name, commands)
        elif 'fish' in shell_name:
            return self._generate_fish_completion(app_name, commands)
        else:
            return None
    
    def _generate_bash_completion(self, app_name, commands):
        """Generate bash completion script"""
        completion_script = f"""
# Bash completion for {app_name}
_{app_name}_completion() {{
    local cur prev commands
    COMPREPLY=()
    cur="${{COMP_WORDS[COMP_CWORD]}}"
    prev="${{COMP_WORDS[COMP_CWORD-1]}}"
    commands="{' '.join(commands)}"
    
    if [[ ${{COMP_CWORD}} == 1 ]]; then
        COMPREPLY=( $(compgen -W "${{commands}}" -- ${{cur}}) )
    fi
}}

complete -F _{app_name}_completion {app_name}
"""
        return {'type': 'bash', 'script': completion_script.strip()}
    
    def _generate_zsh_completion(self, app_name, commands):
        """Generate zsh completion script"""
        completion_script = f"""
#compdef {app_name}

_{app_name}_commands() {{
    local commands
    commands=(
        {' '.join([f"'{cmd}:Execute {cmd} command'" for cmd in commands])}
    )
    _describe 'commands' commands
}}

_{app_name}() {{
    local context state line
    _arguments \\
        '1: :_{app_name}_commands'
}}

_{app_name} "$@"
"""
        return {'type': 'zsh', 'script': completion_script.strip()}
    
    def _generate_fish_completion(self, app_name, commands):
        """Generate fish completion script"""
        completions = []
        for cmd in commands:
            completions.append(f"complete -c {app_name} -a '{cmd}' -d 'Execute {cmd} command'")
        
        completion_script = '\n'.join(completions)
        return {'type': 'fish', 'script': completion_script}
    
    def generate_env_activation(self, env_name, env_path):
        """Generate environment activation script"""
        detection = self.detector.detect_with_fallback()
        if not detection['success']:
            return None
        
        shell_name = detection['shell_name'].lower()
        
        scripts = {}
        
        if 'bash' in shell_name or 'zsh' in shell_name:
            scripts['posix'] = f"""
# Activate {env_name} environment
export PATH="{env_path}/bin:$PATH"
export VIRTUAL_ENV="{env_path}"
export VIRTUAL_ENV_PROMPT="({env_name}) "
PS1="({env_name}) $PS1"

# Deactivation function
deactivate() {{
    export PATH="${{PATH#{env_path}/bin:}}"
    unset VIRTUAL_ENV
    unset VIRTUAL_ENV_PROMPT
    PS1="${{PS1#({env_name}) }}"
    unset deactivate
}}
"""
        
        elif 'fish' in shell_name:
            scripts['fish'] = f"""
# Activate {env_name} environment
set -gx PATH "{env_path}/bin" $PATH
set -gx VIRTUAL_ENV "{env_path}"
set -gx VIRTUAL_ENV_PROMPT "({env_name}) "

function deactivate
    set -e PATH[1]
    set -e VIRTUAL_ENV
    set -e VIRTUAL_ENV_PROMPT
    functions -e deactivate
end
"""
        
        elif 'powershell' in shell_name or 'pwsh' in shell_name:
            scripts['powershell'] = f"""
# Activate {env_name} environment
$env:PATH = "{env_path}\\Scripts;$env:PATH"
$env:VIRTUAL_ENV = "{env_path}"
$env:VIRTUAL_ENV_PROMPT = "({env_name}) "

function deactivate {{
    $env:PATH = $env:PATH -replace "{env_path}\\Scripts;", ""
    Remove-Item Env:VIRTUAL_ENV
    Remove-Item Env:VIRTUAL_ENV_PROMPT
    Remove-Item Function:deactivate
}}
"""
        
        return scripts

# Test development tools integration
dev_tools = DevToolsShellIntegration(prod_detector)

print("1. CLI Completion Generation:")
sample_commands = ['build', 'test', 'deploy', 'status']
completion = dev_tools.generate_cli_completion('myapp', sample_commands)

if completion:
    print(f"   Shell type: {completion['type']}")
    print(f"   Completion script generated: {len(completion['script'])} characters")
    print(f"   Sample: {completion['script'][:100]}...")
else:
    print("   No completion generated (unsupported shell)")

print(f"\n2. Environment Activation:")
env_scripts = dev_tools.generate_env_activation('myproject', '/path/to/myproject')

if env_scripts:
    for shell_type, script in env_scripts.items():
        print(f"   {shell_type.title()} script: {len(script)} characters")
else:
    print("   No activation scripts generated")

# Real-world usage scenarios
print(f"\n=== Real-World Usage Scenarios ===")

usage_scenarios = {
    'Python Package Managers': {
        'tools': ['pip', 'conda', 'poetry', 'pipenv'],
        'use_case': 'Virtual environment activation scripts',
        'shell_dependency': 'High - different syntax per shell'
    },
    'Build Systems': {
        'tools': ['make', 'cmake', 'bazel', 'ninja'],
        'use_case': 'Cross-platform build script execution',
        'shell_dependency': 'Medium - platform-specific commands'
    },
    'CLI Applications': {
        'tools': ['git', 'docker', 'kubectl', 'aws-cli'],
        'use_case': 'Command completion and aliases',
        'shell_dependency': 'High - shell-specific completion syntax'
    },
    'IDE Extensions': {
        'tools': ['VS Code', 'IntelliJ', 'Vim', 'Emacs'],
        'use_case': 'Terminal integration and command execution',
        'shell_dependency': 'Medium - terminal embedding'
    }
}

for scenario, details in usage_scenarios.items():
    print(f"{scenario}:")
    print(f"  Example tools: {', '.join(details['tools'])}")
    print(f"  Use case: {details['use_case']}")
    print(f"  Shell dependency: {details['shell_dependency']}")
    print()

# Configuration and deployment
print(f"=== Configuration and Deployment ===")

config_example = {
    'shell_detection': {
        'enabled': True,
        'cache_duration': 300,
        'fallback_strategies': ['shellingham', 'environment', 'platform_default'],
        'supported_shells': ['bash', 'zsh', 'fish', 'powershell', 'cmd']
    },
    'integration_features': {
        'completion_generation': True,
        'environment_activation': True,
        'command_escaping': True,
        'config_file_detection': True
    },
    'deployment': {
        'auto_detect_on_install': True,
        'generate_completions': True,
        'create_shell_aliases': False,
        'update_config_files': False
    }
}

print("Production configuration example:")
print(json.dumps(config_example, indent=2))

print("\nProduction usage and development tools integration completed!")